# PyTorch Autograd 实验课

自动求导的交互式练习场。配合 `tutorial/02_autograd.py` 学习。

In [13]:
import torch

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.12.1


## 1. requires_grad — 开启梯度追踪

In [14]:
# 默认创建的 tensor 不需要梯度
x = torch.tensor([1.0, 2.0, 3.0])
print(f"默认 requires_grad = {x.requires_grad}")

# 显式声明：我需要梯度
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"声明后 requires_grad = {x.requires_grad}")

# 也可以用 .requires_grad_() 原地设置
y = torch.randn(3).requires_grad_(True)
print(f"y.requires_grad = {y.requires_grad}")

默认 requires_grad = False
声明后 requires_grad = True
y.requires_grad = True


## 2. 前向 → backward → .grad

In [15]:
x = torch.tensor([2.0, 3.0], requires_grad=True)

# 前向：定义计算
y = x ** 2 + 3 * x + 1   # y = x² + 3x + 1
# 手动求导：dy/dx = 2x + 3, 在 x=[2,3] 处应为 [7, 9]

print(f"x = {x}")
print(f"y = {y}   (y = x² + 3x + 1)")

# 反向传播：y 必须是标量！
loss = y.sum()
loss.backward()

print(f"x.grad = {x.grad}    # 手动验证：dy/dx = 2x+3, 在[2,3]处 = [7,9] ✓")

x = tensor([2., 3.], requires_grad=True)
y = tensor([11., 19.], grad_fn=<AddBackward0>)   (y = x² + 3x + 1)
x.grad = tensor([7., 9.])    # 手动验证：dy/dx = 2x+3, 在[2,3]处 = [7,9] ✓


## 3. 链式法则 — 多层计算自动求导

In [16]:
x = torch.tensor(1.0, requires_grad=True)

# 一个稍微复杂的计算：y = sin(x² + 1)
u = x ** 2 + 1      # u = x² + 1
y = torch.sin(u)     # y = sin(u)

y.backward()

# 手动链式法则：dy/dx = cos(u) * 2x
manual_grad = torch.cos(u) * 2 * x
print(f"自动求导 x.grad = {x.grad:.6f}")
print(f"手动链式  2x·cos(x²+1) = {manual_grad:.6f}")
print(f"一致: {torch.allclose(x.grad, manual_grad)} ✓")

自动求导 x.grad = -0.832294
手动链式  2x·cos(x²+1) = -0.832294
一致: True ✓


## 4. 非标量输出 — 传入 gradient 参数

In [24]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2   # y 是向量 [1, 4, 9]，不是标量

# 向量不能直接 backward()，需要告诉 PyTorch 「每个输出的权重」
y_ones_like = torch.ones_like(y)
print(f"ones like = {y_ones_like}")
y.backward(gradient=y_ones_like)

print(f"y = {y}")
print(f"x.grad = {x.grad}   # dy/dx = 2x = [2, 4, 6] ✓")

# 如果 gradient 不是全 1：
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2
y.backward(gradient=torch.tensor([1.0, 0.0, 0.0]))
print(f"gradient=[1,0,0] -> x.grad = {x.grad}   # 只关心第一个输出")

ones like = tensor([1., 1., 1.])
y = tensor([1., 4., 9.], grad_fn=<PowBackward0>)
x.grad = tensor([2., 4., 6.])   # dy/dx = 2x = [2, 4, 6] ✓
gradient=[1,0,0] -> x.grad = tensor([2., 0., 0.])   # 只关心第一个输出


## 5. torch.no_grad() — 推理/评估时关闭追踪

In [25]:
x = torch.tensor(2.0, requires_grad=True)

# 正常计算会建图
y = x ** 2
print(f"正常计算: y.requires_grad = {y.requires_grad}")

# no_grad 上下文里：不建图，省内存，速度快
with torch.no_grad():
    y2 = x ** 2
    print(f"no_grad内: y2.requires_grad = {y2.requires_grad}")
    print(f"         y2 = {y2}  # 值照算，梯度不追踪")

正常计算: y.requires_grad = True
no_grad内: y2.requires_grad = False
         y2 = 4.0  # 值照算，梯度不追踪


## 6. detach() — 从计算图中「摘」出来

In [19]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2          # y 连着 x
z = y.detach()      # z 和 y 值一样，但离开了计算图

print(f"y = {y}, requires_grad = {y.requires_grad}")
print(f"z = {z}, requires_grad = {z.requires_grad}")

y = 9.0, requires_grad = True
z = 9.0, requires_grad = False


## 7. 梯度累积 — 每次 backward 前记得清零

In [20]:
x = torch.tensor(1.0, requires_grad=True)

print("不清零的情况：")
for i in range(3):
    y = x ** 2
    y.backward()
    print(f"  第 {i+1} 次 backward 后 x.grad = {x.grad}")

print(f"\n⚠️ 梯度是累加的！每次 +2.0")

# 正确做法：
print("\n清零的情况：")
x = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    y = x ** 2
    y.backward()
    print(f"  清零后第 {i+1} 次: x.grad = {x.grad}")
    x.grad.zero_()   # ← 清零！

不清零的情况：
  第 1 次 backward 后 x.grad = 2.0
  第 2 次 backward 后 x.grad = 4.0
  第 3 次 backward 后 x.grad = 6.0

⚠️ 梯度是累加的！每次 +2.0

清零的情况：
  清零后第 1 次: x.grad = 2.0
  清零后第 2 次: x.grad = 2.0
  清零后第 3 次: x.grad = 2.0


## 8. 高阶求导 — create_graph=True

In [26]:
x = torch.tensor(2.0, requires_grad=True)

# 一阶导
y = x ** 3          # y = x³
# create_graph=True：保留计算图，以便对梯度再求导
grad1 = torch.autograd.grad(y, x, create_graph=True)[0]
print(f"y = x³")
print(f"一阶导 dy/dx = 3x² = {grad1} (在 x=2 处应为 12) ✓")

# 二阶导：对一阶导再求导
grad2 = torch.autograd.grad(grad1, x)[0]
print(f"二阶导 d²y/dx² = 6x = {grad2} (在 x=2 处应为 12) ✓")

y = x³
一阶导 dy/dx = 3x² = 12.0 (在 x=2 处应为 12) ✓
二阶导 d²y/dx² = 6x = 12.0 (在 x=2 处应为 12) ✓


## 📝 核心速查

| 操作 | 说明 |
|------|------|
| `requires_grad=True` | 开启梯度追踪 |
| `.backward()` | 反向传播，结果存在 `.grad` 里 |
| `.grad` | 梯度值（只有叶子节点才有） |
| `.grad.zero_()` | 清零梯度（每个训练 step 前必做！） |
| `torch.no_grad():` | 暂停追踪（推理/评估用） |
| `.detach()` | 从计算图摘除（值不变） |
| `create_graph=True` | 保留图用于高阶求导 |

**典型训练片段：**
```python
optimizer.zero_grad()   # 清零
loss.backward()         # 反传
optimizer.step()        # 更新参数
```

👉 下一步：去 `practice/02_autograd.py` 刷题！